# 🟡 Lesson 12 — NetCDF4

**Level: Intermediate** · The low-level reader/writer for .nc files — full control over dimensions, variables and metadata.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

In [1]:
import numpy as np
import netCDF4
from pathlib import Path
print("netCDF4", netCDF4.__version__)

GEN = Path("..") / "data" / "generated"; GEN.mkdir(parents=True, exist_ok=True)
path = GEN / "borehole_temps.nc"

netCDF4 1.7.4


## 1. Create a NetCDF file from scratch
A borehole temperature log: dimensions → variables → attributes → data.

In [2]:
nc = netCDF4.Dataset(path, "w", format="NETCDF4")

nc.createDimension("depth", 40)

depth = nc.createVariable("depth", "f4", ("depth",))
temp  = nc.createVariable("temperature", "f4", ("depth",), fill_value=-999.0)

depth.units = "m"; depth.positive = "down"
temp.units = "degC"; temp.long_name = "borehole temperature"
nc.title = "Synthetic geothermal gradient log"
nc.institution = "Python for Geologists lessons"

d = np.linspace(0, 780, 40)
depth[:] = d
temp[:]  = 14.5 + 0.031 * d + np.random.default_rng(1).normal(0, 0.15, 40)

nc.close()
print("wrote", path)

wrote ..\data\generated\borehole_temps.nc


## 2. Read it back — always inspect structure first

In [3]:
nc = netCDF4.Dataset(path)
print("global attrs:", {k: nc.getncattr(k) for k in nc.ncattrs()})
print("dimensions  :", {k: len(v) for k, v in nc.dimensions.items()})
for name, var in nc.variables.items():
    print(f"variable {name:<12} {var.dtype} {var.dimensions} units={var.units}")

global attrs: {'title': 'Synthetic geothermal gradient log', 'institution': 'Python for Geologists lessons'}
dimensions  : {'depth': 40}
variable depth        float32 ('depth',) units=m
variable temperature  float32 ('depth',) units=degC


## 3. Extract data and compute the geothermal gradient

In [4]:
d = nc.variables["depth"][:]
t = nc.variables["temperature"][:]
grad = np.polyfit(d, t, 1)[0] * 1000      # degC per km
print("fitted gradient: %.1f degC/km" % grad)
print("T at bottom (%.0f m): %.1f degC" % (d[-1], t[-1]))
nc.close()

fitted gradient: 31.0 degC/km
T at bottom (780 m): 38.7 degC


## 4. Groups and unlimited dimensions — for growing datasets

In [5]:
path2 = GEN / "monitoring.nc"
nc = netCDF4.Dataset(path2, "w")
nc.createDimension("time", None)                 # unlimited: append forever
st = nc.createGroup("station_UNSA")              # groups = folders inside the file
st.createDimension("time", None)
v = st.createVariable("radon_bq_m3", "f4", ("time",))
v[:] = [55.2, 61.0, 58.7]                        # first three readings
v[3] = 60.1                                      # append a fourth
print("readings stored:", v[:].data.round(1))
nc.close()

readings stored: [55.2 61.  58.7 60.1]


## 5. When to use netCDF4 vs xarray
- **xarray** for analysis (selection, math, plotting) — it *uses* netCDF4 underneath.
- **netCDF4** when you need precise control: unlimited dimensions, groups, compression settings, CF-compliant metadata for publishing data.

### ✏️ Try it
1. Re-open `borehole_temps.nc` with `xr.open_dataset` and compare the workflow.
2. Add `zlib=True, complevel=4` to `createVariable` and compare file sizes.

📚 Docs: https://unidata.github.io/netcdf4-python/